# Human-in-the-Loop: ကြိုတင်လုပ်ဆောင်မှု ပိတ်တံခါးများ၊ စိုးရိမ်မှု အဆင့်ခွဲခြားမှုနှင့် စစ်ဆေးမှတ်တမ်းတင်ခြင်း

ဒီ သင်ခန်းစာအတွက် README သည် Human-in-the-Loop ကို agent သည် တုံ့ပြန်ချက်တစ်ခုရှိပြီးနောက် `APPROVE` သို့မဟုတ် `REJECT` ကို အသုံးပြုသူထံ မေးမြန်းသော စာပိုဒ်တိုတစ်ပိုင်းဖြင့် မိတ်ဆက်ပေးထားသည်။ ထိုနမူနာသည် စတင်ရန်ကောင်းသော နည်းလမ်းဖြစ်သော်လည်း ထုတ်လုပ်မှု HITL အကောင်အထည်ဖော်မှုများတွင် သုံးခု ထပ်မံလိုအပ်သည်။

1. အန္တရာယ်ရှိသော အဆင့်ကို agent တက်ဆောင်မလုပ်ခင်မှာ အလုပ်လုပ်သော **ကြိုတင် လုပ်ဆောင်မှု ပိတ်တံခါး**တစ်ခု၊ ထို့ကြောင့် ကုန်ကျစရိတ်၊ မပြန်လည်ပြောင်းလဲနိုင်မှုနှင့် နောက်ကျမှုများ ထိန်းချုပ်နိုင်စေရန်။
2. **စိုးရိမ်မှု အဆင့်ခွဲခြားခြင်း**၊ အနည်းငယ် အန္တရာယ်ရှိသော လုပ်ဆောင်ချက်များကို အလိုအလျောက် ဆောင်ရွက်စေခြင်း၊ အလယ်အလတ် အန္တရာယ်ရှိသော လုပ်ဆောင်ချက်များကို အစုလိုက် အတည်ပြုမှုရှိစေခြင်းနှင့် အမြင့်ဆုံး အန္တရာယ်ရှိသော လုပ်ဆောင်ချက်များကို လူ့အဖွဲ့ဝင်၏ ဂရုစိုက်မှုအောက်တွင်ပဲ တားဆီးခြင်း။
3. **စစ်ဆေးမှု မှတ်တမ်းနှင့် ပြင်ဆင်မှု လည်ပတ်မှု**တစ်ခု၊ ထို့ကြောင့် ပိတ်တံခါး ဆုံးဖြတ်ချက်တိုင်းကို JSONL အဖြစ် မှတ်တမ်းတင်ပြီး၊ ငြင်းပယ်ခြင်းသည် agent ကို `Revising...` ဟူသောစာမထုတ်ပဲ ဖွဲ့စည်းထားသော ကြောင်းပြချက်ဖြင့် ပြန်လည်မေးမြန်းခြင်း ဖြစ်စေသည်။

ဤ notebook သည် `06-system-message-framework.ipynb` အောက်ခံနည်းလမ်းတူ မူလအခြေအနေများအပေါ်တွင် ထုတ်လုပ်ထားသည်။ ဒါဟာ `DEMO_MODE = True` ဆိုပါက သက်ဆိုင်ရာ အင်တာแက်တစ်တစ်မျိုး မလိုအပ်ဘဲ အဆုံးသတ်အဆင့်အထိ အလုပ်လုပ်မည် ဖြစ်ပြီး `DEMO_MODE = False` ဆိုပါက မျန့်ကြားချက်များကို `input()` ဖြင့် တိုက်ရိုက် မေးမြန်းနိုင်ပါသည်။ မှတ်ချက်- DEMO_MODE တွင် တတိယရည်မှန်းချက်၏ ပြန်ကြားမှုကို စာရင်းလုပ်ထားသောကြောင့် loop အလုပ်လုပ်ပုံကို အဆုံးမှအဆုံးကြည့်ရှုနိုင်သည်။ ရုပ်သိမ်းဖြင့် ပြန်လည်သတ်မှတ်မှုသည် `DEMO_MODE = False` လိုအပ်ပြီး စီမံခန့်ခွဲသူ တစ်ဦး လုပ်ဆောင်ရမည် ဖြစ်သည်။

**အတွင်းမပါ ဝန်ကျ כי (အခြားသင်ခန်းစာများ၌ ကိုင်တွယ်):** စိစစ်မှုနှင့် ဝင်ရောက်ခွင့်ထိန်းချုပ်မှု (သင်ခန်းစာ 06 README အကြောင်းအရာ ၂), ကိရိယာခေါ်ဆိုမှု အလယ်အလတ် (သင်ခန်းစာ 14 MAF စူးစမ်းချက်), အမြောက်အများ အေးဂျင့် ဆွေးနွေးမှုနမူနာများ။

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## ပုံစံ ၁: ကြိုလုပ်ဆောင်မှု အတားအဆီး

README ထဲက HITL ဂဏန်းနည်းလမ်းမှာ အေးဂျင့်ကို ပထမဆုံးခေါ်ပြီး၊ ထို့နောက် အသုံးပြုခွင့်ကို အတည်ပြုဖို့ မေးသည်။ ၎င်းသည် **လုပ်ဆောင်ပြီးနောက်** လမ်းစဉ်ဖြစ်သည်။ အေးဂျင့်သည် မပျက်မကွက် လုပ်ဆောင်ပြီးဖြစ်သောကြောင့် LLM ခေါ်စာရင်းတောင်းခံခြင်း၏ ကုန်ကျစရိတ်ကို ရှေ့ပြေးပေးပြီးသဖြင့်၊ အနှောင့်အယှက် (အီးမေးလ်ပို့ခြင်း၊ ဒေတာဘေ့စ်စာရင်း များရေးသားခြင်း၊ မှတ်ချက် စာတင်ခြင်း) များသည် အပြီးသတ်ဖြစ်ပြီးဖြစ်သည်။

**ကြိုလုပ်ဆောင်မှု** လမ်းစဉ်မှာ အေးဂျင့်သည် အန္တရာယ်ရှိသော ခြေလှမ်းကို လုပ်ဆောင်ရန်မတိုင်မှီ အတားအဆီးကို ထည့်သွင်းသည်။ အေးဂျင့်သည် လုပ်ဆောင်ချက်ကို အကြံပြုပြီး၊ အတားအဆီးသည် ဆောင်ရွက်ရေးပညာရပ်များကို ဆုံးဖြတ်သည်၊ အတည်ပြုချက်ရပြီးမှသာ အနှောင့်အယှက် ဖြစ်ပေါ်သည်။

| လက္ခဏာ | လုပ်ဆောင်ပြီးနောက် အသုံးပြုခွင့် (README ဂဏန်းနည်း) | ကြိုလုပ်ဆောင်မှု အတားအဆီး (ဤ ဒိုင်ယာရီ) |
|---|---|---|
| အသုံးပြုခွင့် ရရှိချိန် | အေးဂျင့် output ထုတ်ပြီးနောက် | အနှောင့်အယှက် ဖြစ်မှီ မတိုင်ခင် |
| ငြင်းဆိုမှုတွင် LLM ကုန်ကျစရိတ် | ရှေ့ပြေးပေးပြီး | အကြံပြုချက်အတွက်သာ ပေးပြီး လုပ်ဆောင်မှုအတွက်မဟုတ် |
| နုတ်ဆက်လွတ်မရသော အနှောင့်အယှက်များ | ဖြစ်နိုင်သည် (လုပ်ဆောင်မှုပြီးဖြစ်) | ကြိုတားဆီးထားသည် |
| စစ်ဆေးမှု ဖော်ပြချက် | အသုံးပြုခွင့်သည် print ကြေညာချက် ဖြစ်သည် | အသုံးပြုခွင့်သည် ရက်စွဲ၊ လုပ်ဆောင်ချက် နှင့် အကြောင်းဖေါ်ပြသော JSON မှတ်တမ်းဖြစ်သည် |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## ပုံစံ ၂: ရှုပ်ထွေးမှုအဆင့်သတ်မှတ်ခြင်း

လူ့အတည်ပြုချက် လိုအပ်သည့် လုပ်ဆောင်ချက်အားလုံးမဟုတ်ပါ။ ပြည်သူ့ API တစ်ခုအား ဖတ်-only လုပ်ဆောင်ခြင်းသည် ဖောက်သည်အီးမေးလ်ပို့ခြင်းထက် ရောထွေးမှုကွာခြားသည်။ နှစ်ခုစလုံးကို တူညီသလို ကိုင်တွယ်ခြင်းသည် အရာထမ်း၏ သတိစူးစိုက်မှုကို အသုံးမတတ်ပစ်ပြီး ကိုယ်စားလှယ်ကို အမြန်မောင်းနှင်နိုင်စေပါ။

ရိုးရှင်းသော ၃-အဆင့် မော်ဒယ် -

| အဆင့် | ဥပမာများ | အတည်ပြုပြုလုပ်မှု လမ်းကြောင်း |
|---|---|---|
| `နိမ့်` (ဖတ်-only) | သတင်းအချက်အလက် အခြေခံ စူးစမ်းခြင်း၊ မီးလျှပ်စစ် လမ်းကြောင်း ရှာဖွေခြင်း၊ ပြည်သူ့ဝက်ဘ်စာမျက်နှာ တင်ယူခြင်း | အလိုအလျောက် လုပ်ဆောင်ပြီး စုံစမ်းစာရင်းအတွက် မှတ်တမ်းတင်ခြင်း |
| `အလယ်အလတ်` (စျေးနှုန်းမကြီးသော ပြောင်းလဲမှု) | ရလဒ်ကို သိမ်းဆည်းခြင်း၊ ကောင်တာတစ်ခု တိုးခြင်း၊ သတိပေးချက် စီစဉ်ခြင်း | အလိုအလျောက် လုပ်ဆောင်သော်လည်း မျက်နှာချင်းဆိုင် ပြန်လည်သုံးသပ်ခြင်းကို တစ်နေ့တစ်ကြိမ် စုစည်းလုပ်ဆောင်ခြင်း |
| `မြင့်မား` (ပြင်ပနှင့် ဆက်နွယ်သော်လည်း မပြန်လည်ပြောင်းလဲနိုင်သော) | အီးမေးလ် ပို့ခြင်း၊ ကဒ် ဘဏ္ဍာရေး လုပ်ဆောင်ခြင်း၊ ပြည်သူ့ချန်နယ်သို့ တင်ခြင်း | လူ့အတည်ပြုချက်ကို ခေတ္တရပ်ဆိုင်းထားသည် |

ဤသည်သည် အဆင့်သတ်မှတ်ခြင်းတစ်မျိုး ဖြစ်သည်။ ထုတ်လုပ်မှု စနစ်များတွင် ပိုမိုသွေမြန်သော အဆင့်များ (ဥပမာ၊ AWS IAM ခွင့်ပြုချက်အဆင့်များ၊ အခန်းကဏ္ဍအခြေခံ ခွင့်ပြုချက်အဆင့်) ကို အသုံးပြုကြသည်။ ေအာက္မှာရှိသော ၃-အဆင့် ဗားရှင်းသည် ဖတ်-only နှင့် လက်တွေ့ ဆောင်ရွက်မှုများကို ပေါင်းစပ်သည့် ကိုယ်စားလှယ်များအတွက် အသုံးဝင်သည့် အနည်းဆုံး ဗားရှင်းဖြစ်သည်။

အောက်ပါ စားစစ်သူသည် စကားလုံးနည်းပညာများကို အသုံးပြုသဖြင့် ဒီမိုကို သက်သာသောနှင့် စိတ်ချရစေရန် ဖြစ်သည်။ ထုတ်လုပ်မှု စနစ်တွင် တိုင်းတာထားသော စားစစ်သူတစ်ခု သို့မဟုတ် ပေါ်လစီအင်ဂျင်တစ်ခုကို လဲသုံးမည် ဖြစ်သည်။

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Pattern 3: Audit log and revision loop

`print("Response approved.")` သည် audit log မဟုတ်ပါ။ ယုံကြည်မှုအတွက်၊ gate တစ်ခုတည်း၏ ဆုံးဖြတ်ချက်ကို structured event အနေနဲ့ မှတ်တမ်းတင်ထားရမယ်၊ ထို event ကိုနောက်မှာ query လုပ်လို့ရ၊ ပြန်လည်ဖတ်ရှုနိုင်ပြီး incident review တွင် ထည့်သွင်းစစ်ဆေးနိုင်ဖို့ ဖြစ်ပါတယ်။

အပိုင်းနှစ်ခုရှိပါတယ်-

1. **Append-only JSONL။** ဆုံးဖြတ်ချက် တစ်ခုစီကို တစ်ကြောင်းချင်းစီ လိုက်ပြီး၊ timestamp, action, tier, decision, reason ကိုပါ ထည့်ရေးထားပါတယ်။ grep လုပ်ရလွယ်ကူပြီး၊ နောက်တစ်ကြောင်းတွင် ဆောင်ရွက်ရန် log store ထဲသို့ ပို့ရလွယ်ပါသည်။
2. **ပြန်လည်သုံးသပ်မှုလှိုင်း (revision loop) ကို ငြင်းပယ်မှုဖြင့် ဖန်တီးသည်။** gate မှ `deny` ပြန်လာလျှင် agent သည် ငြင်းပယ်ခြင်းအကြောင်းပြချက်ကို context အနေနှင့် ထည့်သွင်း၍ ပြန်မေးမြန်းပါသည်၊ ထို့ကြောင့် နောက်တစ်ကြိမ် တင်ပြမှုသည် ပြဿနာကို ရှောင်ကြဉ်နိုင်ပါသည်။

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## အပိုဆောင်းအရင်းအမြစ်များ

အများပြည်သူစီမံကိန်းအချို့သည် ဒီ HITL ပုံစံမျိုးစုံကို အကောင်အထည်ဖော်ထားကြသည်။ သင့်စတက်လုပ်ငန်းအတွက် ကိုက်ညီမှုရှိစေမည့်နည်းလမ်းများကို နှိုင်းယှဉ်ကြည့်ပါ-

- **LangChain** လူကြားပတ်သက်မှုရှိသော ကိရိယာ wrapper များ ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)) - လူသားထည့်သွင်းမှုအတွက် အကောင်အထည်ဖော်ခြင်းကို ရပ်တန့်စေသည့် ကိရိယာ wrapper များ။
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ သည် ဤကို ပြန်လည်စီမံထားသည်) - လူသားကို တိကျစွာဖော်ပြရန် ဒေါင်းလိုက်များနှစ်ပေါင်းပါသော agent ၏ အခန်းကဏ္ဍကို အသုံးပြုသည်။
- **Semantic Kernel** function filters ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)) - မည်သည့် function ခေါ်ဆိုမှု၌မဆို chạy နေသည့် middleware ဖြစ်ပြီး gating လုပ်ဆောင်ချက်များအတွက် သင့်တော်သည်။

တိုင်းစီမံကိန်းများသည် သုံးခုခွဲထားသော sub-pattern များကို မတူညီစွာ ကန့်သတ်ပြုလုပ်သည်။ LangChain သည် tools အဖြစ် wrap လုပ်ထားပြီး၊ AutoGen သည် agent အခန်းကဏ္ဍကို အသုံးပြုသည့်အတွက်၊ Semantic Kernel သည် middleware filter များကို အသုံးပြုသည်။ သင့်သည်ရွေးချယ်ရန်မီ နောက်ဆုံးအထိ နှစ်ခုသို့မဟုတ် တစ်ခုလုံး အကောင်အထည်ဖော်ချက်များကို ဖတ်ရှုစဉ်းစားပါ။

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
